# 03 LightGBM vs. MLP at equal budget
Goal: pick the teacher and the student family fairly. This is not a verdict on trees vs. nets. Both models get the same budget from `configs/config.yaml`: `tune.n_trials` trials and `tune.time_cap_seconds` seconds, whichever runs out first.

In [4]:
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.lib import evaluate_regression, load_data, load_split
from src.utils import load_config, set_seed

cfg = load_config()
set_seed(cfg["seed"])

X, y = load_data()
X, y = X.to_numpy(), y.to_numpy()
_test_idx, folds = load_split()  # same 5 folds as every other model; the test rows are not used here
print("budget:", cfg["tune"])

budget: {'n_trials': 20, 'time_cap_seconds': 900}


## Search loop
One loop serves both models. It logs one row per trial: the trial number, the parameters, CV mean and std, trial seconds and cumulative seconds. It stops after `n_trials`, or after the trial that crosses the time cap.

In [5]:
def run_search(sample_params, cv_rmse, n_trials, time_cap_seconds, seed):
    """Random search. Stops after n_trials, or after the trial that crosses the time cap."""
    rng = np.random.default_rng(seed)
    rows = []
    start = time.perf_counter()
    for trial in range(1, n_trials + 1):
        t0 = time.perf_counter()
        params = sample_params(rng)
        cv_mean, cv_std = cv_rmse(params)
        now = time.perf_counter()
        rows.append(
            {
                "trial": trial,
                **params,
                "cv_rmse_mean": cv_mean,
                "cv_rmse_std": cv_std,
                "trial_seconds": now - t0,
                "cumulative_seconds": now - start,
            }
        )
        if now - start >= time_cap_seconds:
            break
    return pd.DataFrame(rows)

## LightGBM search
Random search. Each trial samples seven settings, then is scored by the same 5-fold CV. Rounds use early stopping on a 10% slice of each fold's *training* rows, so the validation fold never influences training.

In [6]:
def sample_lgbm_params(rng):
    """Draw one random setting. Rates and penalties are sampled on a log scale."""
    return {
        "learning_rate": float(np.exp(rng.uniform(np.log(0.01), np.log(0.2)))),
        "num_leaves": int(rng.integers(8, 129)),
        "n_estimators": int(np.exp(rng.uniform(np.log(200), np.log(2000)))),  # max rounds
        "min_child_samples": int(rng.integers(5, 101)),
        "subsample": float(rng.uniform(0.5, 1.0)),  # row subsampling
        "colsample_bytree": float(rng.uniform(0.5, 1.0)),  # column subsampling
        "reg_lambda": float(np.exp(rng.uniform(np.log(1e-3), np.log(10)))),  # L2 penalty
    }


def cv_rmse_lgbm(params):
    """5-fold CV RMSE for one setting: (mean, std) over the folds."""
    scores = []
    for train_idx, val_idx in folds:
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=cfg["seed"])
        model = lgb.LGBMRegressor(
            **params, subsample_freq=1, random_state=cfg["seed"], verbose=-1
        )
        model.fit(
            X[fit_idx],
            y[fit_idx],
            eval_X=X[stop_idx],
            eval_y=y[stop_idx],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        pred = model.predict(X[val_idx])  # uses the best round found by early stopping
        scores.append(evaluate_regression(y[val_idx], pred)["rmse"])
    return float(np.mean(scores)), float(np.std(scores, ddof=1))

### Tiny run: 3 trials
Checks that the loop works and the log has the right columns. It is not saved. The real search moves to `src/tune.py`.

In [7]:
tiny_lgbm = run_search(
    sample_lgbm_params, cv_rmse_lgbm, 3, cfg["tune"]["time_cap_seconds"], cfg["seed"]
)
tiny_lgbm.round(4)

,trial,learning_rate,num_leaves,n_estimators,min_child_samples,subsample,colsample_bytree,reg_lambda,cv_rmse_mean,cv_rmse_std,trial_seconds,cumulative_seconds
0,1,0.1016,87,1444,47,0.8487,0.5471,7.9889,0.4663,0.0117,28.0260,28.0260
1,2,0.0978,94,268,80,0.7252,0.6854,5.0940,0.4549,0.0120,22.4910,50.5170
2,3,0.0688,56,555,83,0.6136,0.7773,0.0018,0.4606,0.0132,16.9218,67.4389


**Takeaways**
- All three trials score between 0.455 and 0.466, a spread about the size of the fold std (0.012), so the settings can't be ranked yet.
- Each trial takes 17 to 28 s. The log has every column the real search needs.

## MLP search
A PyTorch MLP trained with Adam and MSE loss. Features and target are both standardized, using statistics from the training rows only. Early stopping watches the MSE on the same 10% slice of each fold's training rows that LightGBM uses. Predictions are converted back to units of $100k before computing RMSE, so the score is directly comparable with LightGBM's. Random search over width, depth, learning rate, weight decay and dropout.

In [8]:
import copy

import torch
from sklearn.preprocessing import StandardScaler
from torch import nn

torch.set_num_threads(1)  # LightGBM and torch each bring their own OpenMP runtime; with several
# torch threads the second one deadlocks in this process. One thread is plenty for this small net.
DEVICE = "cpu"  # CPU keeps runs deterministic; record the device next to any timing
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 100, 10, 256
print("device:", DEVICE)


def sample_mlp_params(rng):
    """Draw one random setting. Rates and penalties are sampled on a log scale."""
    return {
        "width": int(rng.choice([32, 64, 128, 256])),
        "depth": int(rng.integers(1, 5)),  # hidden layers
        "learning_rate": float(np.exp(rng.uniform(np.log(1e-4), np.log(1e-2)))),
        "weight_decay": float(np.exp(rng.uniform(np.log(1e-6), np.log(1e-2)))),
        "dropout": float(rng.uniform(0.0, 0.3)),
    }


def build_mlp(n_in, width, depth, dropout):
    layers, size = [], n_in
    for _ in range(depth):
        layers += [nn.Linear(size, width), nn.ReLU(), nn.Dropout(dropout)]
        size = width
    layers.append(nn.Linear(size, 1))
    return nn.Sequential(*layers)


def fit_predict_mlp(params, fit_idx, stop_idx, val_idx):
    """Train on fit_idx, stop early on stop_idx, return predictions for val_idx in $100k."""
    set_seed(cfg["seed"])
    x_scaler = StandardScaler().fit(X[fit_idx])
    y_mean, y_std = y[fit_idx].mean(), y[fit_idx].std()

    def to_x(idx):
        return torch.tensor(x_scaler.transform(X[idx]), dtype=torch.float32)

    def to_y(idx):
        return torch.tensor((y[idx] - y_mean) / y_std, dtype=torch.float32)

    x_fit, y_fit, x_stop, y_stop = to_x(fit_idx), to_y(fit_idx), to_x(stop_idx), to_y(stop_idx)
    model = build_mlp(X.shape[1], params["width"], params["depth"], params["dropout"])
    opt = torch.optim.Adam(
        model.parameters(), lr=params["learning_rate"], weight_decay=params["weight_decay"]
    )
    loss_fn = nn.MSELoss()

    best_loss, best_state, bad_epochs = float("inf"), None, 0
    for _ in range(MAX_EPOCHS):
        model.train()
        order = torch.randperm(len(x_fit))
        for i in range(0, len(order), BATCH_SIZE):
            batch = order[i : i + BATCH_SIZE]
            opt.zero_grad()
            loss_fn(model(x_fit[batch]).squeeze(1), y_fit[batch]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            stop_loss = loss_fn(model(x_stop).squeeze(1), y_stop).item()
        if stop_loss < best_loss - 1e-6:
            best_loss, best_state, bad_epochs = stop_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(to_x(val_idx)).squeeze(1).numpy()
    return pred * y_std + y_mean  # undo the target scaling


def cv_rmse_mlp(params):
    """5-fold CV RMSE for one setting: (mean, std) over the folds."""
    scores = []
    for train_idx, val_idx in folds:
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=cfg["seed"])
        pred = fit_predict_mlp(params, fit_idx, stop_idx, val_idx)
        scores.append(evaluate_regression(y[val_idx], pred)["rmse"])
    return float(np.mean(scores)), float(np.std(scores, ddof=1))

device: cpu


### Tiny run: 3 trials
Same log columns as LightGBM. The RMSE is in units of $100k, so the two tables can be compared directly.

In [9]:
tiny_mlp = run_search(
    sample_mlp_params, cv_rmse_mlp, 3, cfg["tune"]["time_cap_seconds"], cfg["seed"]
)
tiny_mlp.round(4)

,trial,width,depth,learning_rate,weight_decay,dropout,cv_rmse_mean,cv_rmse_std,trial_seconds,cumulative_seconds
0,1,32,4,0.0008,0.0027,0.2092,0.5609,0.0179,6.6802,6.6802
1,2,32,1,0.0089,0.0011,0.2358,0.6041,0.0127,1.0593,7.7396
2,3,128,1,0.0008,0.0000,0.2780,0.5769,0.0159,4.1441,11.8836


**Takeaways** (tiny runs, 3 trials each, so a check that the code works, not a verdict)
- **Both beat the baselines easily.** All six trials score below the depth-6 tree baseline (RMSE 0.693).
- **LightGBM is ahead so far.** Its trials score 0.455 to 0.466, while the MLP's score 0.561 to 0.604. The gap is far larger than the fold std (about 0.015). The MLP only got 3 random settings, though, so this is not yet a fair comparison.
- **Trial cost differs by a lot.** A LightGBM trial took 17 to 28 s and an MLP trial 1 to 7 s, which is why the budget is set in both trials and seconds.
- **The cap may not bind.** At these speeds 20 trials take about 7 minutes for LightGBM and about 2 for the MLP, both under the 900 s cap.